# Building 3D & AR Experiences with the SceneView MCP Server

This notebook shows how to give Claude expert knowledge of the [SceneView](https://github.com/sceneview/sceneview) 3D/AR SDK using the Model Context Protocol (MCP). With the SceneView MCP server connected, Claude can:

1. **Generate correct, compilable 3D/AR code** for Android (Jetpack Compose) and iOS (SwiftUI)
2. **Validate code** against 15+ common mistakes before presenting it
3. **Return interactive 3D previews** as HTML artifacts users can view in the browser
4. **Look up the exact API** for any of 26+ node types

We will demonstrate how to call these MCP tools via the Anthropic Messages API, simulating what happens when Claude Desktop or Claude Code connects to the MCP server.

---

## Prerequisites

- An [Anthropic API key](https://console.anthropic.com/settings/keys)
- Node.js >= 18 (for the MCP server, used in the "Try it yourself" section)

### Quick install for Claude Desktop / Claude Code

If you just want to use the MCP server directly (no Python needed):

**Claude Desktop** — add to `~/Library/Application Support/Claude/claude_desktop_config.json`:
```json
{
  "mcpServers": {
    "sceneview": {
      "command": "npx",
      "args": ["-y", "sceneview-mcp"]
    }
  }
}
```

**Claude Code** — run:
```bash
claude mcp add sceneview -- npx -y sceneview-mcp
```

## Setup

In [ ]:
%pip install anthropic

In [ ]:
import json
import os

import anthropic

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Defining SceneView MCP tools

The SceneView MCP server exposes 12 tools. Below we define a representative subset as Anthropic API tool schemas — the same schemas the MCP server registers. This lets us demonstrate the tool-use pattern without running the MCP server process.

In [ ]:
SCENEVIEW_TOOLS = [
    {
        "name": "get_sample",
        "description": "Returns a complete, compilable Kotlin or Swift sample for a given SceneView scenario. Call list_samples first if unsure which scenario fits.",
        "input_schema": {
            "type": "object",
            "properties": {
                "scenario": {
                    "type": "string",
                    "enum": [
                        "model-viewer",
                        "ar-model-viewer",
                        "ar-augmented-image",
                        "physics-demo",
                        "procedural-geometry",
                        "compose-ui-3d",
                        "ios-model-viewer",
                        "ios-ar-model-viewer",
                    ],
                    "description": "The scenario to fetch.",
                }
            },
            "required": ["scenario"],
        },
    },
    {
        "name": "validate_code",
        "description": "Checks a Kotlin or Swift SceneView snippet for common mistakes: threading violations, wrong destroy order, missing null-checks, LightNode trailing-lambda bug, deprecated 2.x APIs. Always call this before presenting generated SceneView code.",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "The Kotlin or Swift source code to validate.",
                }
            },
            "required": ["code"],
        },
    },
    {
        "name": "get_node_reference",
        "description": "Returns the full API reference for a specific SceneView node type — parameters, types, and usage example.",
        "input_schema": {
            "type": "object",
            "properties": {
                "nodeType": {
                    "type": "string",
                    "description": 'Node type to look up, e.g. "ModelNode", "LightNode", "ARScene".',
                }
            },
            "required": ["nodeType"],
        },
    },
    {
        "name": "create_3d_artifact",
        "description": 'Generates a self-contained HTML page with interactive 3D content. Types: "model-viewer" for 3D model viewing, "chart-3d" for 3D data visualization, "scene" for rich 3D scenes, "product-360" for product turntables with hotspot annotations.',
        "input_schema": {
            "type": "object",
            "properties": {
                "type": {
                    "type": "string",
                    "enum": ["model-viewer", "chart-3d", "scene", "product-360"],
                    "description": "The type of 3D artifact to generate.",
                },
                "modelUrl": {
                    "type": "string",
                    "description": "Public URL to a .glb model file (HTTPS, CORS-enabled).",
                },
                "title": {
                    "type": "string",
                    "description": "Title shown above the 3D viewer.",
                },
                "data": {
                    "type": "array",
                    "description": "Data points for chart-3d type. Array of {label, value, color} objects.",
                    "items": {
                        "type": "object",
                        "properties": {
                            "label": {"type": "string"},
                            "value": {"type": "number"},
                            "color": {"type": "string"},
                        },
                    },
                },
            },
            "required": ["type"],
        },
    },
    {
        "name": "get_setup",
        "description": "Returns the Gradle dependency and AndroidManifest snippet for SceneView.",
        "input_schema": {
            "type": "object",
            "properties": {
                "type": {
                    "type": "string",
                    "enum": ["3d", "ar"],
                    "description": '"3d" for 3D-only scenes. "ar" for augmented reality.',
                }
            },
            "required": ["type"],
        },
    },
]

## Example 1: Generate a 3D model viewer

Ask Claude to build a 3D model viewer. With the tools available, Claude will call `get_sample` to fetch the correct starting code, then `validate_code` to check it for errors.

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": "Build me a Jetpack Compose screen that displays a 3D model of a chair with orbit controls. Use SceneView.",
        }
    ],
)

print(f"Stop reason: {response.stop_reason}")
for block in response.content:
    if block.type == "tool_use":
        print(f"\nTool call: {block.name}")
        print(f"Input: {json.dumps(block.input, indent=2)}")
    elif block.type == "text":
        print(f"\nText: {block.text[:200]}...")

Claude requests the `get_sample` tool with `scenario: "model-viewer"`. In a real MCP setup, the server would return the complete sample code. Let's simulate that response and continue the conversation.

In [ ]:
# Simulate the MCP tool response with a realistic code sample
SAMPLE_MODEL_VIEWER = """@Composable
fun ModelViewerScreen() {
    val modelLoader = rememberModelLoader()
    val modelInstance = rememberModelInstance(modelLoader, "models/chair.glb")

    Scene(
        modifier = Modifier.fillMaxSize(),
        cameraManipulator = rememberCameraManipulator(
            CameraManipulator.Mode.ORBIT
        ),
        environment = rememberEnvironment(modelLoader, "environments/studio.hdr")
    ) {
        modelInstance?.let { instance ->
            ModelNode(
                modelInstance = instance,
                scaleToUnits = 1.0f
            )
        }
    }
}"""

# Find the tool_use block to get its ID
tool_use_block = next(b for b in response.content if b.type == "tool_use")

# Continue the conversation with the tool result
followup = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": "Build me a Jetpack Compose screen that displays a 3D model of a chair with orbit controls. Use SceneView.",
        },
        {"role": "assistant", "content": response.content},
        {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": tool_use_block.id,
                    "content": SAMPLE_MODEL_VIEWER,
                }
            ],
        },
    ],
)

# Print Claude's final response
for block in followup.content:
    if block.type == "text":
        print(block.text)
    elif block.type == "tool_use":
        print(f"\n[Tool call: {block.name}({json.dumps(block.input)[:100]}...)]")

## Example 2: Create an interactive 3D artifact

The `create_3d_artifact` tool generates self-contained HTML that renders an interactive 3D viewer. In Claude Desktop, this appears as a live artifact the user can rotate, zoom, and even view in AR on mobile.

Let's ask Claude to visualize some data in 3D.

In [ ]:
response_3d = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": "Show me a 3D bar chart of Q1 revenue by region: North America $4.2M, Europe $3.1M, Asia $2.8M, Latin America $1.5M.",
        }
    ],
)

for block in response_3d.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Input:\n{json.dumps(block.input, indent=2)}")

Claude calls `create_3d_artifact` with `type: "chart-3d"` and the data points. The MCP server generates a complete HTML page using CSS 3D transforms that renders an interactive 3D bar chart — no WebGL dependencies, works in any browser.

In Claude Desktop, this HTML is rendered inline as a live artifact. The user sees a 3D chart they can rotate by dragging.

## Example 3: Code validation catches real bugs

The `validate_code` tool catches subtle mistakes that even experienced developers make. Let's intentionally introduce a common bug — the LightNode trailing-lambda mistake — and see if Claude catches it.

In [ ]:
BUGGY_CODE = """
@Composable
fun MyScene() {
    Scene(modifier = Modifier.fillMaxSize()) {
        // BUG: LightNode's apply is a named parameter, not a trailing lambda
        LightNode {
            intensity(100_000f)
        }
        ModelNode(
            modelInstance = rememberModelInstance(rememberModelLoader(), "model.glb")
        )
    }
}
"""

response_validate = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    tools=SCENEVIEW_TOOLS,
    messages=[
        {
            "role": "user",
            "content": f"Check this SceneView code for bugs:\n```kotlin\n{BUGGY_CODE}\n```",
        }
    ],
)

for block in response_validate.content:
    if block.type == "tool_use":
        print(f"Tool: {block.name}")
        print(f"Claude sends the code to validate_code for automated checking.")
    elif block.type == "text":
        print(block.text)

The validator catches two issues:

1. **LightNode trailing lambda** — `LightNode { ... }` should be `LightNode(apply = { ... })`. This is a named parameter, not a trailing lambda. Using the wrong syntax compiles but silently does nothing.

2. **Missing null check** — `rememberModelInstance()` returns `ModelInstance?` (nullable). The model hasn't loaded yet on the first composition. Passing it directly to `ModelNode` without a null check crashes.

These are exactly the kind of SDK-specific bugs that generic AI models get wrong. The MCP server encodes this domain knowledge.

## Agentic loop: full generate-validate-fix cycle

In practice, Claude uses the tools in an agentic loop:

1. `get_sample` or `get_node_reference` — fetch the right starting code or API docs
2. Generate code based on the user's request
3. `validate_code` — check for mistakes
4. Fix any issues found
5. Present the validated code

Here is a helper that runs this loop automatically.

In [ ]:
def run_sceneview_agent(user_prompt: str, tool_results: dict | None = None) -> str:
    """Run a multi-turn tool-use loop with SceneView tools.

    Args:
        user_prompt: The user's request.
        tool_results: Optional dict mapping tool names to mock responses
                      for demonstration without the live MCP server.

    Returns:
        Claude's final text response after all tool calls resolve.
    """
    tool_results = tool_results or {}
    messages = [{"role": "user", "content": user_prompt}]

    for _ in range(5):  # max 5 tool-use turns
        response = client.messages.create(
            model="claude-haiku-4-5",
            max_tokens=4096,
            tools=SCENEVIEW_TOOLS,
            messages=messages,
        )

        # If Claude is done (no tool calls), return the text
        if response.stop_reason == "end_turn":
            return "\n".join(
                block.text for block in response.content if block.type == "text"
            )

        # Process tool calls
        messages.append({"role": "assistant", "content": response.content})

        tool_result_contents = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  -> Tool call: {block.name}({list(block.input.keys())})")
                # Use mock result or a default message
                result = tool_results.get(
                    block.name,
                    f"Tool {block.name} executed successfully. (Mock response for demo)",
                )
                tool_result_contents.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    }
                )

        messages.append({"role": "user", "content": tool_result_contents})

    return "Max tool-use turns reached."


print("Agent helper defined. See the next cell for usage.")

In [ ]:
# Demonstrate the agentic loop with a mock tool response
result = run_sceneview_agent(
    "What Gradle setup do I need for a SceneView AR project?",
    tool_results={
        "get_setup": """// build.gradle.kts (Module: app)
dependencies {
    implementation("io.github.sceneview:arsceneview:3.3.0")
}

// AndroidManifest.xml
<uses-permission android:name="android.permission.CAMERA" />
<uses-feature android:name="android.hardware.camera.ar" android:required="true" />
<application>
    <meta-data android:name="com.google.ar.core" android:value="required" />
</application>""",
    },
)

print("\n--- Claude's response ---")
print(result[:500])

## What users can ask

With the SceneView MCP server connected, Claude handles prompts like:

| Prompt | Tools used |
|--------|------------|
| "Build me a 3D model viewer in Compose" | `get_sample` -> `validate_code` |
| "Add AR plane detection to my app" | `get_ar_setup` -> `get_sample(ar-model-viewer)` |
| "Show me what a chair looks like in 3D" | `create_3d_artifact(model-viewer)` |
| "What's the API for LightNode?" | `get_node_reference(LightNode)` |
| "Migrate my SceneView 2.x code to 3.0" | `get_migration_guide` |
| "Create a 3D revenue chart" | `create_3d_artifact(chart-3d)` |
| "Set up SceneView for iOS with SwiftUI" | `get_ios_setup` -> `get_sample(ios-model-viewer)` |

The MCP server handles all the domain expertise — threading rules, Filament JNI constraints, the `LightNode` named-parameter gotcha, nullable model loading — so Claude generates correct code on the first try.

## Summary

The SceneView MCP server demonstrates a pattern for giving Claude deep SDK expertise:

- **Domain-specific tools** (`get_sample`, `validate_code`, `get_node_reference`) encode the knowledge that generic training data lacks
- **Code validation as a tool** catches bugs before the user sees them — a "linter in the loop" pattern applicable to any SDK
- **Interactive artifacts** (`create_3d_artifact`) let Claude generate visual outputs, not just text
- **The tool-use loop** (fetch reference -> generate -> validate -> fix) ensures correctness

This approach works for any SDK or framework: package the API docs, common patterns, and a validator into an MCP server, and Claude becomes an expert.

### Resources

- [SceneView GitHub](https://github.com/sceneview/sceneview)
- [sceneview-mcp on npm](https://www.npmjs.com/package/sceneview-mcp)
- [Model Context Protocol docs](https://modelcontextprotocol.io/)
- [Anthropic tool use guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview)